In [1]:
# Board composition measured as at 31 October each year
# Source: FTSE Women Leaders Review Portal
# Note: timing differs from fiscal year-end financials — discuss in methodology

In [2]:
import pandas as pd
import numpy as np
import os

# create folders if they don't exist
for folder in ["data/raw", "data/processed", "outputs/figures", "outputs/tables"]:
    os.makedirs(folder, exist_ok=True)

print("Folders ready")
print("pandas:", pd.__version__)

Folders ready
pandas: 2.2.3


In [3]:
url = "https://ftsewomenleaders.com/company-rankings/"

tables = pd.read_html(url)

print(f"Number of tables found: {len(tables)}")
for i, t in enumerate(tables):
    print(f"\nTable {i}: shape {t.shape}")
    print("Columns:", t.columns.tolist())

Number of tables found: 3

Table 0: shape (95, 8)
Columns: ['Rank', 'Company', 'Supersector', 'Women on Boards', 'Board Size', 'Total Women on Boards', 'Executive Women on Boards', 'Combined Exec.Comm & DRs']

Table 1: shape (165, 8)
Columns: ['Rank', 'Company', 'Supersector', 'Women on Boards', 'Board Size', 'Total Women on Boards', 'Executive Women on Boards', 'Combined Exec.Comm & DRs']

Table 2: shape (48, 8)
Columns: ['Rank', 'Company', 'Supersector', 'Women on Boards', 'Board Size', 'Total Women on Boards', 'Executive Women on Boards', 'Combined Exec.Comm & DRs']


In [4]:
ftse100 = tables[0]

print("Shape:", ftse100.shape)
print("\nData types:")
print(ftse100.dtypes)
print("\nFirst 10 rows:")
ftse100.head(10)

Shape: (95, 8)

Data types:
Rank                          int64
Company                      object
Supersector                  object
Women on Boards              object
Board Size                    int64
Total Women on Boards         int64
Executive Women on Boards     int64
Combined Exec.Comm & DRs     object
dtype: object

First 10 rows:


,Rank,Company,Supersector,Women on Boards,Board Size,Total Women on Boards,Executive Women on Boards,Combined Exec.Comm & DRs
0,1,Burberry Group Plc,Consumer Products & Services,55.6%,9,5,1,56.4%
1,2,Next Plc,Retail,33.3%,12,4,1,53.6%
2,3,Marks & Spencer Group Plc,"Personal Care, Drug & Grocery Stores",66.7%,9,6,1,50.0%
3,4,Phoenix Group Holdings Plc,Insurance,58.3%,12,7,0,49.5%
4,5,Convatec Group Plc,Health Care,55.6%,9,5,1,49.4%
5,6,Diageo Plc,"Food, Beverage & Tobacco",77.8%,9,7,1,49.1%
6,7,Haleon Plc,Health Care,58.3%,12,7,1,47.8%
7,8,Pearson Plc,Media,63.6%,11,7,1,47.6%
8,9,AstraZeneca Plc,Health Care,50.0%,14,7,1,46.8%
9,10,National Grid Plc,Utilities,41.7%,12,5,1,45.7%


In [5]:
print(ftse100.tail(5))
print("\nMissing values per column:")
print(ftse100.isnull().sum())

    Rank                   Company                   Supersector  \
90    91     JD Sports Fashion Plc                        Retail   
91    92        Intertek Group Plc   Industrial Goods & Services   
92    93           Antofagasta Plc               Basic Resources   
93    94         Ashtead Group Plc   Industrial Goods & Services   
94    95  Games Workshop Group Plc  Consumer Products & Services   

   Women on Boards  Board Size  Total Women on Boards  \
90           30.0%          10                      3   
91           30.8%          13                      4   
92           30.8%          13                      4   
93           22.2%           9                      2   
94           28.6%           7                      2   

    Executive Women on Boards Combined Exec.Comm & DRs  
90                          0                    32.6%  
91                          0                    27.7%  
92                          0                    24.7%  
93                  

In [6]:
df = ftse100.copy()

# Convert percentage strings to floats
df["WomenPct"] = df["Women on Boards"].str.replace("%", "", regex=False).astype(float)
df["ExecCommPct"] = df["Combined Exec.Comm & DRs"].str.replace("%", "", regex=False).astype(float)

# Rename to clean column names
df = df.rename(columns={
    "Board Size": "BoardSize",
    "Total Women on Boards": "TotalWomen",
    "Executive Women on Boards": "ExecWomen"
})

# Tag the year — this is the 2026 report, reporting on 2025 data
df["Year"] = 2025

# Standardised company key for later merging
df["CompanyKey"] = (df["Company"]
                    .str.upper()
                    .str.replace(r"\b(PLC|LIMITED|LTD|GROUP|HOLDINGS)\b", "", regex=True)
                    .str.replace(r"[^A-Z0-9 ]", "", regex=True)
                    .str.replace(r"\s+", " ", regex=True)
                    .str.strip())

keep = ["CompanyKey", "Company", "Year", "Supersector", "WomenPct", 
        "BoardSize", "TotalWomen", "ExecWomen", "ExecCommPct"]
df_clean = df[keep]

print(df_clean.dtypes)
print()
df_clean.head(10)

CompanyKey      object
Company         object
Year             int64
Supersector     object
WomenPct       float64
BoardSize        int64
TotalWomen       int64
ExecWomen        int64
ExecCommPct    float64
dtype: object



,CompanyKey,Company,Year,Supersector,WomenPct,BoardSize,TotalWomen,ExecWomen,ExecCommPct
0,BURBERRY,Burberry Group Plc,2025,Consumer Products & Services,55.6,9,5,1,56.4
1,NEXT,Next Plc,2025,Retail,33.3,12,4,1,53.6
2,MARKS SPENCER,Marks & Spencer Group Plc,2025,"Personal Care, Drug & Grocery Stores",66.7,9,6,1,50.0
3,PHOENIX,Phoenix Group Holdings Plc,2025,Insurance,58.3,12,7,0,49.5
4,CONVATEC,Convatec Group Plc,2025,Health Care,55.6,9,5,1,49.4
5,DIAGEO,Diageo Plc,2025,"Food, Beverage & Tobacco",77.8,9,7,1,49.1
6,HALEON,Haleon Plc,2025,Health Care,58.3,12,7,1,47.8
7,PEARSON,Pearson Plc,2025,Media,63.6,11,7,1,47.6
8,ASTRAZENECA,AstraZeneca Plc,2025,Health Care,50.0,14,7,1,46.8
9,NATIONAL GRID,National Grid Plc,2025,Utilities,41.7,12,5,1,45.7


In [7]:
print(df_clean["WomenPct"].describe())
print("\nFirms below 30% (critical mass threshold):", (df_clean["WomenPct"] < 30).sum())
print("Firms at or above 30%:", (df_clean["WomenPct"] >= 30).sum())
print("\nBoard size range:", df_clean["BoardSize"].min(), "to", df_clean["BoardSize"].max())

count    95.000000
mean     44.510526
std       9.582603
min      22.200000
25%      39.250000
50%      44.400000
75%      50.000000
max      77.800000
Name: WomenPct, dtype: float64

Firms below 30% (critical mass threshold): 4
Firms at or above 30%: 91

Board size range: 7 to 18


In [8]:
df_clean.to_csv("data/raw/board_2025_clean.csv", index=False)
print(f"Saved {len(df_clean)} rows")

Saved 95 rows


In [9]:
import requests
import os

os.makedirs("data/raw/ftse_reports", exist_ok=True)

pdfs = {
    "ftse_wl_2024.pdf": "https://ftsewomenleaders.com/wp-content/uploads/2025/03/ftse-report-master-2025-online-v3.pdf",
    "ftse_wl_2023.pdf": "https://ftsewomenleaders.com/wp-content/uploads/2024/04/ftse-women-leaders-report-final-april-2024.pdf",
    "ftse_wl_2022.pdf": "https://ftsewomenleaders.com/wp-content/uploads/2023/03/ftse-women-leaders-review-report-2022-v2.pdf",
    "ftse_wl_2021.pdf": "https://ftsewomenleaders.com/wp-content/uploads/2022/03/2021_FTSE-Women-Leaders-Review_Final-Report_WA.pdf",
    "ftse_wl_2020.pdf": "https://ftsewomenleaders.com/wp-content/uploads/2021/03/Hampton-Alexander-Review-Report-2020_web.pdf",
    "ftse_wl_2019.pdf": "https://ftsewomenleaders.com/wp-content/uploads/2019/11/HA-Review-Report-2019.pdf",
}

for filename, url in pdfs.items():
    path = f"data/raw/ftse_reports/{filename}"
    if os.path.exists(path):
        print(f"Already have {filename}")
        continue
    r = requests.get(url, timeout=60)
    with open(path, "wb") as f:
        f.write(r.content)
    print(f"Downloaded {filename} ({len(r.content)/1e6:.1f} MB)")

Already have ftse_wl_2024.pdf
Already have ftse_wl_2023.pdf
Already have ftse_wl_2022.pdf
Already have ftse_wl_2021.pdf
Already have ftse_wl_2020.pdf
Already have ftse_wl_2019.pdf


In [10]:
!pip install pdfplumber
import pdfplumber

with pdfplumber.open("data/raw/ftse_reports/ftse_wl_2024.pdf") as pdf:
    print("Total pages:", len(pdf.pages))
    for i, page in enumerate(pdf.pages):
        t = page.extract_text() or ""
        if "FTSE 100 Rankings" in t:
            print(f"PDF page index {i} contains FTSE 100 Rankings")

Defaulting to user installation because normal site-packages is not writeable
Total pages: 72
PDF page index 41 contains FTSE 100 Rankings
PDF page index 42 contains FTSE 100 Rankings
PDF page index 43 contains FTSE 100 Rankings
PDF page index 44 contains FTSE 100 Rankings


In [15]:
import pdfplumber
import re
import pandas as pd

ROW_RE = re.compile(r"^(\d{1,3})\s+(.+?)\s+(\d{1,3}\.\d)%\s+(.*?)\s*(\d{1,3}\.\d)%$")

def parse_column(text, year, rows):
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    i = 0
    while i < len(lines):
        if re.match(r"^\d{1,3}\s", lines[i]):
            matched = False
            for span in (1, 2, 3):
                if i + span > len(lines):
                    break
                candidate = " ".join(lines[i:i+span])
                m = ROW_RE.match(candidate)
                if m:
                    rank, name_sector, women_pct, roles, exco_pct = m.groups()
                    rows.append({"Year": year, "Rank": int(rank), "NameSector": name_sector.strip(), "WomenPct": float(women_pct), "KeyRoles": roles.strip(), "ExCoPct": float(exco_pct)})
                    i += span
                    matched = True
                    break
            if not matched:
                i += 1
        else:
            i += 1

def extract_ftse100(pdf_path, year, page_range):
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num in page_range:
            page = pdf.pages[page_num]
            mid = page.width / 2
            parse_column(page.crop((0, 0, mid, page.height)).extract_text() or "", year, rows)
            parse_column(page.crop((mid, 0, page.width, page.height)).extract_text() or "", year, rows)
    return pd.DataFrame(rows).sort_values("Rank").reset_index(drop=True)

df_2024 = extract_ftse100("data/raw/ftse_reports/ftse_wl_2024.pdf", 2024, range(41, 45))
print(f"Extracted {len(df_2024)} rows")
print(f"Rank range: {df_2024['Rank'].min()} to {df_2024['Rank'].max()}")
print(f"Duplicates: {df_2024['Rank'].duplicated().sum()}")
missing = sorted(set(range(1, 97)) - set(df_2024['Rank']))
print(f"Missing ranks: {missing}")
df_2024.head(15)

Extracted 96 rows
Rank range: 1 to 96
Duplicates: 0
Missing ranks: []


,Year,Rank,NameSector,WomenPct,KeyRoles,ExCoPct
0,2024,1,Marks & Spencer Group Plc,60.0,SID CFO,54.0
1,2024,2,Pearson Plc Media,60.0,CFO,51.5
2,2024,3,Next Plc Retail,33.3,,50.5
3,2024,4,"Diageo Plc Food, Beverage & Tobacco",70.0,SID CEO,49.5
4,2024,5,AstraZeneca Plc Health Care,42.9,CFO,48.9
5,2024,6,Phoenix Group Holdings Plc Insurance,41.7,SID CFO,47.5
6,2024,7,National Grid Plc Utilities,36.4,Chair,47.2
7,2024,8,Haleon Plc Health Care,54.5,CFO,46.8
8,2024,9,NatWest Group Plc Banks,50.0,CFO,46.8
9,2024,10,BP Plc Energy,54.5,SID CFO,46.5


In [16]:
SECTORS = [
    "Personal Care, Drug & Grocery Stores", "Industrial Goods & Services",
    "Consumer Products & Services", "Food, Beverage & Tobacco",
    "Telecommunications", "Financial Services", "Travel & Leisure",
    "Basic Resources", "Health Care", "Real Estate", "Technology",
    "Insurance", "Utilities", "Chemicals", "Energy", "Media",
    "Retail", "Banks", "Automobiles & Parts", "Construction & Materials"
]

def split_name_sector(text):
    for sector in sorted(SECTORS, key=len, reverse=True):
        if text.endswith(sector):
            return text[:-len(sector)].strip(), sector
    return text.strip(), None

df_2024[["Company", "Sector"]] = df_2024["NameSector"].apply(lambda x: pd.Series(split_name_sector(x)))

print("Rows with no sector matched:", df_2024["Sector"].isna().sum())
print()
print(df_2024[df_2024["Sector"].isna()][["Rank", "NameSector"]])

Rows with no sector matched: 11

    Rank                   NameSector
0      1    Marks & Spencer Group Plc
27    28              J Sainsbury Plc
28    29     Barratt Developments Plc
29    30                 Unilever Plc
44    45            Compass Group Plc
55    56                Persimmon Plc
57    58  Berkeley Group Holdings Plc
70    71                    Tesco Plc
81    82  Reckitt Benckiser Group Plc
86    87            Taylor Wimpey Plc
91    92     Games Workshop Group Plc


In [17]:
def make_key(s):
    return (s.str.upper()
             .str.replace(r"\b(PLC|LIMITED|LTD|GROUP|HOLDINGS|SA|AG|S\.A\.)\b", "", regex=True)
             .str.replace(r"[^A-Z0-9 ]", "", regex=True)
             .str.replace(r"\s+", " ", regex=True)
             .str.strip())

df_2024["CompanyKey"] = make_key(df_2024["Company"])

df_2024[["Rank", "Company", "CompanyKey", "Sector", "WomenPct"]].head(10)

,Rank,Company,CompanyKey,Sector,WomenPct
0,1,Marks & Spencer Group Plc,MARKS SPENCER,None,60.0
1,2,Pearson Plc,PEARSON,Media,60.0
2,3,Next Plc,NEXT,Retail,33.3
3,4,Diageo Plc,DIAGEO,"Food, Beverage & Tobacco",70.0
4,5,AstraZeneca Plc,ASTRAZENECA,Health Care,42.9
5,6,Phoenix Group Holdings Plc,PHOENIX,Insurance,41.7
6,7,National Grid Plc,NATIONAL GRID,Utilities,36.4
7,8,Haleon Plc,HALEON,Health Care,54.5
8,9,NatWest Group Plc,NATWEST,Banks,50.0
9,10,BP Plc,BP,Energy,54.5


In [18]:
df_2024.to_csv("data/raw/board_2024_clean.csv", index=False)
print(f"Saved {len(df_2024)} rows")

Saved 96 rows


In [20]:
for year in [2023, 2022, 2021, 2020, 2019]:
    path = f"data/raw/ftse_reports/ftse_wl_{year}.pdf"
    try:
        with pdfplumber.open(path) as pdf:
            pages = [i for i, p in enumerate(pdf.pages) if "FTSE 100 Rankings" in (p.extract_text() or "")]
            print(f"{year}: {len(pdf.pages)} pages total, FTSE 100 Rankings on {pages}")
    except Exception as e:
        print(f"{year}: ERROR — {e}")

2023: 82 pages total, FTSE 100 Rankings on [2, 6]
2022: 82 pages total, FTSE 100 Rankings on [2, 49, 51]
2021: 78 pages total, FTSE 100 Rankings on [2, 47, 49]
2020: 84 pages total, FTSE 100 Rankings on [2, 53, 55, 56]
2019: 80 pages total, FTSE 100 Rankings on [2, 35, 49, 51, 52]


In [21]:
import re

pat = re.compile(r"^\d{1,3}\s+.+?\s+\d{1,3}\.\d%")

for year in [2023, 2022, 2021, 2020, 2019]:
    path = f"data/raw/ftse_reports/ftse_wl_{year}.pdf"
    with pdfplumber.open(path) as pdf:
        hits = []
        for i, p in enumerate(pdf.pages):
            lines = (p.extract_text() or "").split("\n")
            n = sum(1 for l in lines if pat.match(l.strip()))
            if n >= 5:
                hits.append((i, n))
    print(f"{year}: {hits}")

2023: [(11, 8), (13, 5), (28, 6), (36, 9), (40, 10), (49, 25), (50, 26), (51, 27), (52, 17), (53, 27), (54, 25), (55, 25), (56, 26), (57, 21), (58, 22), (59, 13), (61, 10), (62, 5), (64, 12), (65, 22), (66, 23), (67, 25), (68, 30), (69, 21), (70, 26), (71, 20), (72, 19), (73, 29), (74, 26), (75, 34), (76, 25)]
2022: [(11, 9), (19, 7), (30, 10), (49, 25), (50, 26), (51, 25), (52, 16), (53, 25), (54, 25), (55, 25), (56, 24), (57, 24), (58, 21), (59, 15), (61, 5), (62, 7), (65, 24), (66, 23), (67, 21), (68, 28), (69, 18), (70, 21), (71, 20), (72, 23), (73, 31), (74, 29), (75, 30), (76, 20)]
2021: [(15, 6), (19, 5), (21, 6), (30, 6), (47, 23), (48, 26), (49, 26), (50, 19), (51, 26), (52, 26), (53, 23), (54, 23), (55, 24), (56, 20), (57, 22), (58, 6), (59, 6), (61, 18), (63, 20), (64, 24), (65, 18), (66, 25), (67, 27), (68, 26), (69, 24), (70, 31), (71, 27), (72, 22), (73, 21)]
2020: [(15, 6), (30, 6), (53, 23), (54, 26), (55, 26), (56, 21), (57, 14), (58, 22), (59, 11), (60, 26), (61, 25),

In [23]:
for year in [2023, 2022, 2021, 2020, 2019]:
    path = f"data/raw/ftse_reports/ftse_wl_{year}.pdf"
    print(f"\n{'='*20} {year} {'='*20}")
    with pdfplumber.open(path) as pdf:
        for i in range(40, min(len(pdf.pages), 82)):
            text = pdf.pages[i].extract_text() or ""
            first_lines = " | ".join(text.split("\n")[:3])
            if "FTSE" in first_lines or "Appendix" in first_lines or "Ranking" in first_lines:
                print(f"p{i}: {first_lines[:110]}")


==================== 2023 ====================
p40: Document under Embargo until one minute past midnight (00.01pm) on the 27th February 2024 Document under Embar
p41: Document under Embargo until one minute past midnight (00.01pm) on the 27th February 2024 Document under Embar
p45: Document under Embargo until one minute past midnight (00.01pm) on the 27th February 2024 Document under Embar
p47: Document under Embargo until one minute past midnight (00.01pm) on the 27th February 2024 Document under Embar
p48: Document under Embargo until one minute past midnight (00.01pm) on the 27th February 2024 Document under Embar
p49: Document under Embargo until one minute past midnight (00.01pm) on the 27th February 2024 Document under Embar
p51: Document under Embargo until one minute past midnight (00.01pm) on the 27th February 2024 Document under Embar
p53: Document under Embargo until one minute past midnight (00.01pm) on the 27th February 2024 Document under Embar
p55: Document under Emba

In [25]:
with pdfplumber.open("data/raw/ftse_reports/ftse_wl_2023.pdf") as pdf:
    for i in [49, 50, 51]:
        page = pdf.pages[i]
        mid = page.width / 2
        left = page.crop((0, 0, mid, page.height)).extract_text() or ""
        print(f"\n{'='*25} PAGE {i} LEFT COLUMN {'='*25}")
        print(left[:800])


========================= PAGE 49 LEFT COLUMN =========================
Document under Embargo until one minute pas
5. Appendix C FTSE 100
Rankings 2023 Women on Boards and in Lead
Women on Boards data as at 11th January 2024, Leadership da
(Excludes 3 Investment Trusts)
At or above 33% and on
At or above 40% Target
track to meet 40% Target
Wom
Rank28 Company Sector
on Bo
Consumer Products
1 Burberry Group Plc 50
& Services
Personal Care, Drug
2 Marks & Spencer Group Plc 54.
& Grocery Stores
3 Next Plc Retail 36.
4 National Grid Plc Utilities 41.
5 Lloyds Banking Group Plc Banks 45.
6 Pearson Plc Media 54.
7 AstraZeneca Plc Health Care 46.
Food, Beverage
8 Diageo Plc 70
& Tobacco
9 Haleon Plc Health Care 45.
10 BP Plc Energy 50
50
11 Rightmove Plc Real Estate 57.
12 Phoenix Group Holdings Plc Insurance 38.
Personal Care, Drug
13 J Sainsbury Plc 44.
& Grocery S

========================= PAGE 50 LEFT COLUMN =========================
Document under Embargo until one minute pas
Wom
Rank 

In [27]:
import pdfplumber
import re
import pandas as pd

ROW_2023 = re.compile(r"^(\d{1,3})\s+(.+?)\s+(\d{1,3}(?:\.\d)?)%?\s*$")

def parse_col_2023(text, year, rows):
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    for idx, line in enumerate(lines):
        m = ROW_2023.match(line)
        if not m:
            continue
        rank, body, pct = m.groups()
        rank = int(rank)
        if not (1 <= rank <= 110):
            continue
        val = float(pct)
        if not (0 <= val <= 100):
            continue
        rows.append({"Year": year, "Rank": rank, "NameSector": body.strip(), "WomenPct": val})

def extract_2023style(pdf_path, year, page_range, split=0.55):
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for pn in page_range:
            page = pdf.pages[pn]
            mid = page.width * split
            parse_col_2023(page.crop((0, 0, mid, page.height)).extract_text() or "", year, rows)
            parse_col_2023(page.crop((mid, 0, page.width, page.height)).extract_text() or "", year, rows)
    df = pd.DataFrame(rows)
    return df.drop_duplicates(subset=["Rank"]).sort_values("Rank").reset_index(drop=True) if len(df) else df

df_2023 = extract_2023style("data/raw/ftse_reports/ftse_wl_2023.pdf", 2023, range(49, 54))
print(f"Rows: {len(df_2023)}")
if len(df_2023):
    print(f"Ranks {df_2023['Rank'].min()}-{df_2023['Rank'].max()}, missing: {sorted(set(range(1,98)) - set(df_2023['Rank']))}")

Rows: 97
Ranks 1-97, missing: []


In [28]:
print(df_2023.head(15))
print()
print(df_2023["WomenPct"].describe())

    Year  Rank                            NameSector  WomenPct
0   2023     1                    Burberry Group Plc      50.0
1   2023     2             Marks & Spencer Group Plc      54.5
2   2023     3                       Next Plc Retail      36.4
3   2023     4           National Grid Plc Utilities      41.7
4   2023     5        Lloyds Banking Group Plc Banks      45.5
5   2023     6                     Pearson Plc Media      54.5
6   2023     7           AstraZeneca Plc Health Care      46.2
7   2023     8                            Diageo Plc      70.0
8   2023     9                Haleon Plc Health Care      45.5
9   2023    10                         BP Plc Energy      50.0
10  2023    11             Rightmove Plc Real Estate      57.1
11  2023    12  Phoenix Group Holdings Plc Insurance      38.5
12  2023    13                       J Sainsbury Plc      44.4
13  2023    14                    Financial Services      41.7
14  2023    15                 Beazley Plc Insurance   

In [29]:
for year, pages in {2022: range(49, 54), 2021: range(47, 52), 2020: range(53, 58), 2019: range(63, 68)}.items():
    try:
        df = extract_2023style(f"data/raw/ftse_reports/ftse_wl_{year}.pdf", year, pages)
        if len(df) == 0:
            print(f"{year}: 0 rows")
            continue
        print(f"{year}: {len(df)} rows, ranks {df['Rank'].min()}-{df['Rank'].max()}, mean {df['WomenPct'].mean():.1f}%")
        globals()[f"df_{year}"] = df
    except Exception as e:
        print(f"{year}: FAILED — {e}")

2022: 97 rows, ranks 1-97, mean 40.7%
2021: 98 rows, ranks 1-98, mean 39.4%
2020: 97 rows, ranks 1-100, mean 30.0%
2019: 1 rows, ranks 100-100, mean 61.3%


In [30]:
with pdfplumber.open("data/raw/ftse_reports/ftse_wl_2019.pdf") as pdf:
    for i in [63, 64, 65]:
        page = pdf.pages[i]
        print(f"\n{'='*25} PAGE {i} (full width) {'='*25}")
        print((page.extract_text() or "")[:900])


========================= PAGE 63 (full width) =========================
Appendix E
FTSE 350 Sector Analysis
Women on Boards data as at October 2019, Leadership data as at 30th June 2019
Sector : Investment Trusts
FTSE Women Combined
Rank Company Sector Detail
List on Boards Exec.Comm & DRs
1 Law Debenture Corporation Closed End Investments 250 29% 44.9%
2 Scottish Investment Trust Plc Closed End Investments 250 33% 44.4%
3 Syncona LTD Closed End Investments 250 25% 43.8%
4 Caledonia Investments Plc Closed End Investments 250 22% 28.6%
5 BBGI Sicav SA Closed End Investments 250 33% 25%
Average 31.7% 40.1%
Sector Average
Sector : Personal Goods
FTSE Women Combined
Rank Company Sector Detail
List on Boards Exec.Comm & DRs
1 Burberry Group Plc Personal Goods 100 50% 61.3%
2 Unilever Plc Personal Goods 100 38% 33.8%
3 Games Workshop Group Plc Leisure Goods 250 50% 27.3%
4 PZ Cussons Plc Personal Goods 250 43% 25%
New Entry
5 Watches Of Switzerland Group Plc Perso

========================

In [31]:
print(df_2023.head(8))
print()
print("Marks & Spencer:", df_2023[df_2023["NameSector"].str.contains("Marks", na=False)]["WomenPct"].values)

   Year  Rank                      NameSector  WomenPct
0  2023     1              Burberry Group Plc      50.0
1  2023     2       Marks & Spencer Group Plc      54.5
2  2023     3                 Next Plc Retail      36.4
3  2023     4     National Grid Plc Utilities      41.7
4  2023     5  Lloyds Banking Group Plc Banks      45.5
5  2023     6               Pearson Plc Media      54.5
6  2023     7     AstraZeneca Plc Health Care      46.2
7  2023     8                      Diageo Plc      70.0

Marks & Spencer: [54.5]


In [32]:
print(df_2023[df_2023["NameSector"].str.contains("Marks", na=False)][["Rank", "NameSector", "WomenPct"]])

   Rank                 NameSector  WomenPct
1     2  Marks & Spencer Group Plc      54.5


In [33]:
with pdfplumber.open("data/raw/ftse_reports/ftse_wl_2019.pdf") as pdf:
    for i in [63, 64, 65]:
        page = pdf.pages[i]
        print(f"\n{'='*25} PAGE {i} {'='*25}")
        print((page.extract_text() or "")[:900])


========================= PAGE 63 =========================
Appendix E
FTSE 350 Sector Analysis
Women on Boards data as at October 2019, Leadership data as at 30th June 2019
Sector : Investment Trusts
FTSE Women Combined
Rank Company Sector Detail
List on Boards Exec.Comm & DRs
1 Law Debenture Corporation Closed End Investments 250 29% 44.9%
2 Scottish Investment Trust Plc Closed End Investments 250 33% 44.4%
3 Syncona LTD Closed End Investments 250 25% 43.8%
4 Caledonia Investments Plc Closed End Investments 250 22% 28.6%
5 BBGI Sicav SA Closed End Investments 250 33% 25%
Average 31.7% 40.1%
Sector Average
Sector : Personal Goods
FTSE Women Combined
Rank Company Sector Detail
List on Boards Exec.Comm & DRs
1 Burberry Group Plc Personal Goods 100 50% 61.3%
2 Unilever Plc Personal Goods 100 38% 33.8%
3 Games Workshop Group Plc Leisure Goods 250 50% 27.3%
4 PZ Cussons Plc Personal Goods 250 43% 25%
New Entry
5 Watches Of Switzerland Group Plc Perso

========================= PAGE 64 ===

In [34]:
print("Missing 2020 ranks:", sorted(set(range(1, 101)) - set(df_2020["Rank"])))
print(df_2020["WomenPct"].describe())

Missing 2020 ranks: [39, 68, 76]
count    97.000000
mean     30.040206
std       7.944215
min      10.600000
25%      24.300000
50%      30.200000
75%      34.500000
max      53.800000
Name: WomenPct, dtype: float64


In [35]:
import re

ROW_2019 = re.compile(r"^(\d{1,3})\s+(.+?)\s+(100|250)\s+(\d{1,3})%\s+([\d.]+)%\s*$")

def extract_2019(pdf_path, year=2019):
    rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for pn in range(len(pdf.pages)):
            text = pdf.pages[pn].extract_text() or ""
            for line in text.split("\n"):
                m = ROW_2019.match(line.strip())
                if m:
                    rank, body, ftse_list, women, exco = m.groups()
                    rows.append({
                        "Year": year,
                        "SectorRank": int(rank),
                        "NameSector": body.strip(),
                        "FTSEList": int(ftse_list),
                        "WomenPct": float(women),
                        "ExCoPct": float(exco)
                    })
    return pd.DataFrame(rows)

raw_2019 = extract_2019("data/raw/ftse_reports/ftse_wl_2019.pdf")
print(f"Total rows parsed: {len(raw_2019)}")
print(raw_2019["FTSEList"].value_counts())

df_2019 = raw_2019[raw_2019["FTSEList"] == 100].drop_duplicates(subset=["NameSector"]).reset_index(drop=True)
print(f"\nFTSE 100 firms: {len(df_2019)}")
print(f"Mean WomenPct: {df_2019['WomenPct'].mean():.1f}%")
df_2019.head(15)

Total rows parsed: 283
FTSEList
250    184
100     99
Name: count, dtype: int64

FTSE 100 firms: 99
Mean WomenPct: 32.4%


,Year,SectorRank,NameSector,FTSEList,WomenPct,ExCoPct
0,2019,1,Burberry Group Plc Personal Goods,100,50.0,61.3
1,2019,2,Unilever Plc Personal Goods,100,38.0,33.8
2,2019,1,Next Plc Diversified Retailers,100,44.0,53.9
3,2019,9,JD Sports Fashion Plc Apparel Retailers,100,29.0,30.0
4,2019,10,Kingfisher Plc Home Improvement Retailers,100,44.0,29.8
5,2019,14,Just Eat Plc Consumer Services,100,30.0,23.1
6,2019,2,Astrazeneca Plc Pharmaceuticals,100,33.0,40.3
7,2019,3,GlaxoSmithKline Plc Pharmaceuticals,100,45.0,38.1
8,2019,4,Hikma Pharmaceuticals Plc Pharmaceuticals,100,27.0,23.2
9,2019,1,WM Morrison Supermarkets Plc Food Retailers an...,100,22.0,41.4


In [36]:
def clean_company_2019(text):
    m = re.match(r"^(.+?(?:Plc|PLC|Ltd|LTD|LIMITED|SA|S\.A\.|AG))\b", text)
    return m.group(1).strip() if m else text.strip()

df_2019["Company"] = df_2019["NameSector"].apply(clean_company_2019)
df_2019["CompanyKey"] = make_key(df_2019["Company"])
print(df_2019[["Company", "CompanyKey", "WomenPct"]].head(15))

                         Company                CompanyKey  WomenPct
0             Burberry Group Plc                  BURBERRY      50.0
1                   Unilever Plc                  UNILEVER      38.0
2                       Next Plc                      NEXT      44.0
3          JD Sports Fashion Plc         JD SPORTS FASHION      29.0
4                 Kingfisher Plc                KINGFISHER      44.0
5                   Just Eat Plc                  JUST EAT      30.0
6                Astrazeneca Plc               ASTRAZENECA      33.0
7            GlaxoSmithKline Plc           GLAXOSMITHKLINE      45.0
8      Hikma Pharmaceuticals Plc     HIKMA PHARMACEUTICALS      27.0
9   WM Morrison Supermarkets Plc  WM MORRISON SUPERMARKETS      22.0
10   Sainsburys Supermarkets Ltd   SAINSBURYS SUPERMARKETS      30.0
11               Ocado Group Plc                     OCADO      25.0
12                     Tesco Plc                     TESCO      31.0
13                       ITV Plc  

In [37]:
print("FTSE 100 firms extracted:", len(df_2019))
print("Mean WomenPct:", round(df_2019["WomenPct"].mean(), 1))

FTSE 100 firms extracted: 99
Mean WomenPct: 32.4


In [38]:
import pandas as pd

# 2025 from the web scrape — rename to match
df_2025 = pd.read_csv("data/raw/board_2025_clean.csv")
df_2025 = df_2025.rename(columns={"CompanyKey": "CompanyKey"})[["CompanyKey", "Company", "Year", "WomenPct"]]

frames = [df_2025]

for year in [2024, 2023, 2022, 2021, 2020]:
    df = globals()[f"df_{year}"].copy()
    if "Company" not in df.columns:
        df[["Company", "Sector"]] = df["NameSector"].apply(lambda x: pd.Series(split_name_sector(x)))
    if "CompanyKey" not in df.columns:
        df["CompanyKey"] = make_key(df["Company"])
    frames.append(df[["CompanyKey", "Company", "Year", "WomenPct"]])

frames.append(df_2019[["CompanyKey", "Company", "Year", "WomenPct"]])

board_panel = pd.concat(frames, ignore_index=True)
board_panel = board_panel.dropna(subset=["CompanyKey", "WomenPct"])
board_panel = board_panel[board_panel["CompanyKey"] != ""]
board_panel = board_panel.drop_duplicates(subset=["CompanyKey", "Year"])

print(f"Total observations: {len(board_panel)}")
print(f"Unique firms: {board_panel['CompanyKey'].nunique()}")
print()
print(board_panel.groupby("Year").agg(n=("WomenPct", "size"), mean_pct=("WomenPct", "mean")).round(1))

Total observations: 660
Unique firms: 196

       n  mean_pct
Year              
2019  99      32.4
2020  94      30.0
2021  94      39.3
2022  93      40.6
2023  93      42.8
2024  92      44.8
2025  95      44.5


In [39]:
board_panel.to_csv("data/processed/board_diversity_panel.csv", index=False)
print("Saved.")

Saved.


In [40]:
counts = board_panel.groupby("CompanyKey")["Year"].count().sort_values()
print("Firms appearing in only 1 year:", (counts == 1).sum())
print("Firms appearing in all 7 years:", (counts == 7).sum())
print()
print("Singletons (likely matching failures):")
print(counts[counts == 1].index.tolist()[:60])

Firms appearing in only 1 year: 80
Firms appearing in all 7 years: 28

Singletons (likely matching failures):
['10 GLAXOSMITHKLINE', '4 2', 'AIRLINES', 'ADMIRAL NONLIFE', 'ASHTEAD SUPPORT SERVICES', 'ASSOCIATED BRITISH FOODS FOOD PRODUCERS', 'ANTOFAGASTA MINING', 'ANGLO AMERICAN MINING', 'BAE SYSTEMS AEROSPACE DEFENSE', 'BARRATT REDROW', 'BHP', 'BHP BILLITON', 'AVIVA LIFE', 'BABCOCK INTERNATIONAL', 'BM EUROPEAN VALUE RETAIL GENERAL RETAILERS', 'BHP MINING', 'BT PLC29', 'COCACOLA EUROPACIFIC PARTNERS', 'COCACOLA HBC BEVERAGES', 'DCC SUPPORT SERVICES', 'DIAGEO BEVERAGES', 'COCACOLA HBC AG', 'BUNZL GENERAL INDUSTRIALS', 'CARNIVAL', 'BRITISH AMERICAN TOBACCO TOBACCO', 'BT PLC33', 'BURBERRY PERSONAL GOODS', 'BRITISH LAND CO', 'BP OIL GAS INDUSTRY', 'EVRAZ MINING', 'ELECTROCOMPONENTS', 'DS SMITH GENERAL INDUSTRIALS', 'INTERNATIONAL CONSOLIDATED AIRLINES SA', 'INTERTEK SUPPORT SERVICES', 'JD SPORTS FASHION GENERAL RETAILERS', 'KINGFISHER GENERAL RETAILERS', 'JUST EAT', 'JUST EAT TAKEAWAYCOM N

In [43]:
import re

def make_key(s):
    s = s.astype(str)
    s = s.str.replace(r"\d+", "", regex=True)                     # strip footnote digits
    s = s.str.upper()
    s = s.str.replace(r"[^A-Z ]", " ", regex=True)                # punctuation to space
    s = s.str.replace(r"\b(PLC|LIMITED|LTD|GROUP|HOLDINGS|COMPANY|CO|SA|AG|NV|SE|INTERNATIONAL|THE)\b", " ", regex=True)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

In [44]:
SECTOR_WORDS = [
    "PERSONAL GOODS","GENERAL RETAILERS","FOOD PRODUCERS","SUPPORT SERVICES","MINING",
    "BEVERAGES","TOBACCO","GENERAL INDUSTRIALS","AEROSPACE DEFENSE","OIL GAS INDUSTRY",
    "LIFE","NONLIFE","CASINOS AND GAMBLING","GOODS HOME","LIFE INSURANCE","PHARMACEUTICALS",
    "FOOD RETAILERS AND WHOLESALERS","DIVERSIFIED RETAILERS","APPAREL RETAILERS",
    "SPECIALTY RETAILERS","HOME IMPROVEMENT RETAILERS","CONSUMER SERVICES","PUBLISHING",
    "RADIO AND TV BROADCASTERS","MEDIA AGENCIES","ENTERTAINMENT","CLOSED END INVESTMENTS",
    "LEISURE GOODS","CONSUMER DIGITAL SERVICES","ENERGY METALS","BANKS","INSURANCE",
    "UTILITIES","CHEMICALS","REAL ESTATE","TECHNOLOGY","MEDIA","RETAIL","ENERGY",
    "HEALTH CARE","FINANCIAL SERVICES","TRAVEL LEISURE","BASIC RESOURCES"
]

def strip_sector(key):
    changed = True
    while changed:
        changed = False
        for w in sorted(SECTOR_WORDS, key=len, reverse=True):
            if key.endswith(" " + w) or key == w:
                key = key[: -len(w)].strip()
                changed = True
    return key.strip()

board_panel["CompanyKey"] = make_key(board_panel["Company"]).apply(strip_sector)
print("Unique firms now:", board_panel["CompanyKey"].nunique())

Unique firms now: 145


In [45]:
RENAMES = {
    "GLAXOSMITHKLINE": "GSK",
    "ROYAL DUTCH SHELL": "SHELL",
    "ROYAL BANK OF SCOTLAND": "NATWEST",
    "ELECTROCOMPONENTS": "RS",
    "BHP BILLITON": "BHP",
    "COCACOLA HBC": "COCACOLA HBC",
    "JUST EAT TAKEAWAYCOM": "JUST EAT",
    "SAINSBURYS SUPERMARKETS": "J SAINSBURY",
    "WM MORRISON SUPERMARKETS": "MORRISON",
    "BM EUROPEAN VALUE RETAIL": "BM EUROPEAN VALUE RETAIL",
    "SPIRAXSARCO ENGINEERING": "SPIRAX",
    "STANDARD LIFE ABERDEEN": "ABRDN",
    "BARRATT DEVELOPMENTS": "BARRATT REDROW",
}

board_panel["CompanyKey"] = board_panel["CompanyKey"].replace(RENAMES)

counts = board_panel.groupby("CompanyKey")["Year"].count().sort_values()
print("Unique firms:", board_panel["CompanyKey"].nunique())
print("In all 7 years:", (counts == 7).sum())
print("In 1 year only:", (counts == 1).sum())
print()
print("Remaining singletons:")
print(counts[counts == 1].index.tolist())

Unique firms: 137
In all 7 years: 63
In 1 year only: 22

Remaining singletons:
['', 'ADMIRAL NON', 'AIRLINES', 'B M EUROPEAN VALUE', 'COCA COLA EUROPACIFIC PARTNERS', 'CARNIVAL', 'BABCOCK', 'CONSOLIDATED AIRLINES S A', 'ICG', 'ROYAL MAIL', 'NMC HEALTH', 'MORRISON', 'METLEN', 'OCADO FOOD DRUG RETAILERS', 'JUST EAT TAKEAWAY COM', 'JUST EAT', 'PENNON', 'SAINSBURY J', 'RSA INSURANCE NON', 'RSA', 'SPIRAX SARCO ENGINEERING INDUSTRIAL ENGINEERING', 'TUI']


In [46]:
EXTRA_SECTORS = ["FOOD DRUG RETAILERS", "INDUSTRIAL ENGINEERING", "NON", "NONLIFE", "S A"]

def strip_extra(key):
    changed = True
    while changed:
        changed = False
        for w in sorted(EXTRA_SECTORS, key=len, reverse=True):
            if key.endswith(" " + w):
                key = key[:-len(w)].strip()
                changed = True
    return key.strip()

board_panel["CompanyKey"] = board_panel["CompanyKey"].apply(strip_extra)

FINAL_MAP = {
    "SAINSBURY J": "J SAINSBURY",
    "JUST EAT TAKEAWAY COM": "JUST EAT",
    "B M EUROPEAN VALUE": "BM EUROPEAN VALUE RETAIL",
    "CONSOLIDATED AIRLINES": "INTERNATIONAL CONSOLIDATED AIRLINES",
    "AIRLINES": "INTERNATIONAL CONSOLIDATED AIRLINES",
    "SPIRAX SARCO ENGINEERING": "SPIRAX",
}
board_panel["CompanyKey"] = board_panel["CompanyKey"].replace(FINAL_MAP)

# drop empties and any duplicate firm-years created by merging
board_panel = board_panel[board_panel["CompanyKey"].str.len() > 2]
board_panel = board_panel.drop_duplicates(subset=["CompanyKey", "Year"])

counts = board_panel.groupby("CompanyKey")["Year"].count()
print(f"Observations: {len(board_panel)}")
print(f"Unique firms: {board_panel['CompanyKey'].nunique()}")
print(f"In all 7 years: {(counts == 7).sum()}")
print(f"In 4+ years: {(counts >= 4).sum()}")
print()
print(board_panel.groupby("Year").agg(n=("WomenPct","size"), mean=("WomenPct","mean")).round(1))

Observations: 635
Unique firms: 125
In all 7 years: 63
In 4+ years: 87

       n  mean
Year          
2019  96  32.3
2020  91  30.0
2021  89  38.9
2022  89  40.5
2023  89  42.4
2024  89  44.8
2025  92  44.7


In [47]:
board_panel.to_csv("data/processed/board_diversity_panel.csv", index=False)
print("Saved:", board_panel.shape)

Saved: (635, 4)
